# VK CF finetune с logQ-коррекцией

Аналог `tiger/cf_finetune_vk.ipynb`, но с logQ-коррекцией только по негативам (см. `tiger/cf_finetune.ipynb`, функция `nt_xent_loss_with_logq_neg_only`). Идея — в логит-матрице вычитаем `log Q(j)` из всех негативных колонок, а диагональ (true positive) оставляем как есть. Это даёт несмещённый softmax-знаменатель по подвыборке негативов и не штрафует true-positive за популярность.

In [1]:
from collections import defaultdict

import numpy as np
import json
import pickle

In [2]:
base_dir = '../data/VK_small/'
embeddings_input_path = base_dir + 'content_embeddings.pkl'
pairs_path = base_dir + 'positive_pairs.txt'
item_frequencies_path = base_dir + 'item_frequencies.txt'
tuned_embeddings_output_path = base_dir + 'logq_tuned_content_embeddings.pkl'

In [3]:
with open(embeddings_input_path, 'rb') as f:
    data = pickle.load(f)

item_ids = np.array(data['item_id'], dtype=np.int64)
X = np.array(data['embedding'], dtype=np.float32)
X.shape

(11491, 64)

## Чтение пар

In [4]:
pairs = []
with open(pairs_path, 'r', encoding='utf-8') as f:
    for line in f:
        two_ints = line.strip().split()
        if len(two_ints) != 2:
            raise ValueError(f'not two ints, {two_ints}')
        anchor, positive = map(int, two_ints)
        pairs.append((anchor, positive))
len(pairs)

347429

In [5]:
cnt = 0
for (a, b) in pairs:
    if a == b:
        cnt += 1
cnt

0

## Базовый Embedding (frozen)

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import List, Tuple

In [7]:
num_items, D = X.shape
num_items, D

(11491, 64)

In [8]:
max_id = int(item_ids.max())
max_id

11490

In [9]:
base_emb = nn.Embedding(max_id + 1, D)
with torch.no_grad():
    base_emb.weight.zero_()
    base_emb.weight[item_ids] = F.normalize(torch.tensor(X), dim=1)

for p in base_emb.parameters():
    p.requires_grad_(False)

## Tower MLP

In [10]:
class TowerMLP(nn.Module):
    def __init__(self, d_in, d_hidden, d_out, num_layers=2, p_drop=0.0):
        super().__init__()
        layers = []
        last = d_in
        for _ in range(num_layers - 1):
            layers += [nn.Linear(last, d_hidden), nn.ReLU(), nn.Dropout(p_drop)]
            last = d_hidden
        layers += [nn.Linear(last, d_out)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return F.normalize(self.net(x), dim=1)

tower = TowerMLP(D, D, D, num_layers=2, p_drop=0.0)

## Dataset / loader

In [11]:
@dataclass
class PairDataset(torch.utils.data.Dataset):
    pairs: List[Tuple[int, int]]
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        a, p = self.pairs[idx]
        return torch.tensor(a), torch.tensor(p)

In [12]:
B = 32
ds = PairDataset(pairs)
loader = torch.utils.data.DataLoader(ds, batch_size=B, shuffle=True, drop_last=True)

## NT-Xent loss с logQ-коррекцией (только негативы)

In [13]:
def nt_xent_loss_with_logq_neg_only(z1, z2, a_ids, p_ids, logq, tau=0.07):
    B = z1.size(0)
    device = z1.device
    idx = torch.arange(B, device=device)

    # z1 -> z2
    base12 = (z1 @ z2.T) / tau            # [B, B]
    col_logq = logq[p_ids].to(device)     # [B]

    logits12 = base12 - col_logq.unsqueeze(0)  # вычли logq из всех столбцов
    logits12[idx, idx] = base12[idx, idx]      # диагональ (true positive) возвращаем без logq

    # z2 -> z1
    base21 = (z2 @ z1.T) / tau
    row_logq = logq[a_ids].to(device)

    logits21 = base21 - row_logq.unsqueeze(0)
    logits21[idx, idx] = base21[idx, idx]

    labels = idx
    return 0.5 * (F.cross_entropy(logits12, labels) +
                  F.cross_entropy(logits21, labels))

## Тренировка

In [14]:
opt = torch.optim.AdamW(tower.parameters(), lr=3e-4, weight_decay=1e-4)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tower = tower.to(device)
base_emb = base_emb.to(device)
device

/kernel/lib/python3.11/site-packages/_distutils_hack/__init__.py:15: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
/kernel/lib/python3.11/site-packages/_distutils_hack/__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


device(type='cuda')

## Загрузка частот и подсчёт `logq`

In [15]:
counts_list = []
with open(item_frequencies_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        counts_list.append(int(line))
counts = torch.tensor(counts_list, dtype=torch.float32, device=device)

if counts.numel() != max_id + 1:
    raise ValueError('Some item_ids popularities missed')

total_count = counts.sum()
q = counts / torch.clamp(total_count, min=1.0)
logq = torch.log(torch.clamp(q, min=1e-12))
print(f'num_items={counts.numel()}, total_count={int(total_count.item())}')
logq.shape

num_items=11491, total_count=396196


torch.Size([11491])

In [16]:
from tqdm.auto import tqdm

In [17]:
for epoch in range(4):
    running = 0.0
    print(f'len(loader) is {len(loader)}')
    for a_ids, p_ids in tqdm(loader, desc=f'epoch {epoch+1}', leave=False):
        a_ids, p_ids = a_ids.to(device), p_ids.to(device)
        with torch.no_grad():
            a_base = base_emb(a_ids)
            p_base = base_emb(p_ids)
        zA = tower(a_base)
        zP = tower(p_base)
        loss = nt_xent_loss_with_logq_neg_only(zA, zP, a_ids, p_ids, logq, tau=0.07)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(tower.parameters(), 1.0)
        opt.step()
        running += loss.item()
    print(f'epoch {epoch+1}: loss={running/len(loader):.4f}')

len(loader) is 10857


epoch 1: loss=12.1738
len(loader) is 10857


epoch 2: loss=11.9827
len(loader) is 10857


epoch 3: loss=11.9188
len(loader) is 10857


epoch 4: loss=11.8886


## Сохранение тюненых эмбедов

In [18]:
new_df = {
    'item_id': [],
    'embedding': []
}

tower.eval()
with torch.no_grad():
    for batch_ids in tqdm(torch.split(torch.tensor(item_ids), 512)):
        batch_ids = batch_ids.to(device)
        base_vecs = base_emb(batch_ids)
        tuned_vecs = tower(base_vecs)
        new_df['item_id'] += batch_ids.cpu().tolist()
        new_df['embedding'] += tuned_vecs.cpu().tolist()

100%|██████████| 23/23 [00:00<00:00, 740.72it/s]


In [19]:
with open(tuned_embeddings_output_path, 'wb') as f:
    pickle.dump(new_df, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f'tuned_content_embeddings.pkl сохранён: {tuned_embeddings_output_path}')

tuned_content_embeddings.pkl сохранён: ../data/VK_small/logq_tuned_content_embeddings.pkl
